# `ffmpeg` Interface

The `mediatools` package maintains a custom interface for using the ffmpeg library. Here I provide some usage examples.

In [1]:
import mediatools

#### Load Test Data
First, let's download some public video files for demonstration.

In [2]:
from pathlib import Path

# get environment variables for testing
import pydantic_settings
class Settings(pydantic_settings.BaseSettings):
    test_zip_file_url: str
settings = Settings()

# download zip file and dump it into a temporary directory
from example_helpers import RemoteZipDownloader
zd = RemoteZipDownloader.from_url(settings.test_zip_file_url)
temp_path = zd.path

ex_vid = temp_path / "totk_builds/op_builds.mp4"
ex_vid

PosixPath('/tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4')

## Probe

The ability to probe video metadata is one of the most commonly used features of `ffmpeg`. The `probe` function returns a `ProbeInfo` instance that encapsulates the json metadata you would normally receive.

In [3]:
probe = mediatools.ffmpeg.probe(ex_vid)
probe

ProbeInfo(fname='/tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4', nb_streams=2, nb_programs=0, format_name='mov,mp4,m4a,3gp,3g2,mj2', format_long_name='QuickTime / MOV', start_time='0.000000', bit_rate=623790, dur='33.436735', size=2607188, probe_score=100, tags={'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2avc1mp41', 'encoder': 'Lavf58.39.101'}, video_streams=[VideoStreamInfo(stream_ind=0, codec_name='h264', codec_long_name='H.264 / AVC / MPEG-4 AVC / MPEG-4 part 10', start_time='0.000000', start_pts=0, time_base='1/15360', tags={'language': 'und', 'handler_name': 'ISO Media file produced by Google Inc.', 'vendor_id': '[0][0][0][0]'}, disposition={'default': True, 'dub': False, 'original': False, 'comment': False, 'lyrics': False, 'karaoke': False, 'forced': False, 'hearing_impaired': False, 'visual_impaired': False, 'clean_effects': False, 'attached_pic': False, 'timed_thumbnails': False, 'non_diegetic': False, 'captions': False, 'descriptions': False, 'me

For convenience, you can also use `VideoFile.probe` if you are already working with a `VideoFile` instance.

In [4]:
ex_vf = mediatools.VideoFile.from_path(ex_vid)
ex_vf.probe()

ProbeInfo(fname='/tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4', nb_streams=2, nb_programs=0, format_name='mov,mp4,m4a,3gp,3g2,mj2', format_long_name='QuickTime / MOV', start_time='0.000000', bit_rate=623790, dur='33.436735', size=2607188, probe_score=100, tags={'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2avc1mp41', 'encoder': 'Lavf58.39.101'}, video_streams=[VideoStreamInfo(stream_ind=0, codec_name='h264', codec_long_name='H.264 / AVC / MPEG-4 AVC / MPEG-4 part 10', start_time='0.000000', start_pts=0, time_base='1/15360', tags={'language': 'und', 'handler_name': 'ISO Media file produced by Google Inc.', 'vendor_id': '[0][0][0][0]'}, disposition={'default': True, 'dub': False, 'original': False, 'comment': False, 'lyrics': False, 'karaoke': False, 'forced': False, 'hearing_impaired': False, 'visual_impaired': False, 'clean_effects': False, 'attached_pic': False, 'timed_thumbnails': False, 'non_diegetic': False, 'captions': False, 'descriptions': False, 'me

### Basic Metadata

The `tags` property maintains metadata that is attached to the video. This could vary according to the camera used or any converter/translation software applied to the video.

In [5]:
probe.tags

{'major_brand': 'isom',
 'minor_version': '512',
 'compatible_brands': 'isomiso2avc1mp41',
 'encoder': 'Lavf58.39.101'}

In [6]:
probe.bit_rate, probe.duration, probe.size, probe.probe_score, probe.resolution_str()

(623790, 33.436735, 2607188, 100, '360x640')

### Streams

Each video maintains multiple video or audio streams, depending on the type of video file. The `streams` attribute allows you to explore the streams. Here you can see we have one video stream and one audio stream.

In [7]:
len(probe.video_streams), len(probe.audio_streams), len(probe.other_streams)

(1, 1, 0)

You can get the first video stream (in the order it appears in the original file) using the `video` attribute. This is convenient because most video file types have only one video stream.

In [8]:
probe.video

VideoStreamInfo(stream_ind=0, codec_name='h264', codec_long_name='H.264 / AVC / MPEG-4 AVC / MPEG-4 part 10', start_time='0.000000', start_pts=0, time_base='1/15360', tags={'language': 'und', 'handler_name': 'ISO Media file produced by Google Inc.', 'vendor_id': '[0][0][0][0]'}, disposition={'default': True, 'dub': False, 'original': False, 'comment': False, 'lyrics': False, 'karaoke': False, 'forced': False, 'hearing_impaired': False, 'visual_impaired': False, 'clean_effects': False, 'attached_pic': False, 'timed_thumbnails': False, 'non_diegetic': False, 'captions': False, 'descriptions': False, 'metadata': False, 'dependent': False, 'still_image': False}, height=640, width=360, coded_width=360, coded_height=640, bits_per_raw_sample=8, avg_frame_rate='30/1', r_frame_rate='30/1', chroma_location='left', color_range='tv', color_space='bt709', field_order='progressive', has_b_frames=1, closed_captions=False, is_avc='true', level=30, pix_fmt='yuv420p', profile='Main', refs=1)

In [9]:
probe.video.resolution, probe.video.codec_name, probe.video.start_time, probe.video.frame_rate

((360, 640), 'h264', '0.000000', 30)

Similarly, the first audio channel can be retrieved using `audio` attribute.

In [10]:
probe.audio

AudioStreamInfo(stream_ind=1, codec_name='aac', codec_long_name='AAC (Advanced Audio Coding)', start_time='0.000000', start_pts=0, time_base='1/44100', tags={'language': 'eng', 'handler_name': 'ISO Media file produced by Google Inc.', 'vendor_id': '[0][0][0][0]'}, disposition={'default': True, 'dub': False, 'original': False, 'comment': False, 'lyrics': False, 'karaoke': False, 'forced': False, 'hearing_impaired': False, 'visual_impaired': False, 'clean_effects': False, 'attached_pic': False, 'timed_thumbnails': False, 'non_diegetic': False, 'captions': False, 'descriptions': False, 'metadata': False, 'dependent': False, 'still_image': False}, sample_fmt='fltp', sample_rate=44100, channels=2, channel_layout='stereo')

In [11]:
probe.audio.codec_name, probe.audio.start_time, probe.audio.channels, probe.audio.channel_layout, probe.audio.sample_rate

('aac', '0.000000', 2, 'stereo', 44100)

## `ffmpeg` Command Interface

The low-level ffmpeg interface allows you to access the full range of ffmpeg features through a pythonic interface.

In [12]:
from mediatools.ffmpeg import ffmpeg, ffinput, ffoutput

### Executing ffmpeg Commands

Create the command object using the `ffmpeg` function along with `ffinput` and `ffoutput`.

This example shows a command that will cut a video halfway through. You would instantiate the command as follows:

In [49]:
output_path = temp_path / "test_video_output.mp4"
half_duration = probe.duration / 2
print(f'{half_duration=}')

cmd = ffmpeg(
    input = ffinput(ex_vid, ss=probe.start_time, to=half_duration),
    output = ffoutput(output_path, y=True),
)
cmd.get_command()

half_duration=16.7183675


'ffmpeg -ss 0.000000 -to 16.7183675 -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -y /tmp/tmpn7bpy8zq/test_video_output.mp4'

`VideoFile` instances also have an `ffmpeg` method that is very similar except that the input is the single video file and you must specify input arguments using the `FFInputArgs` type.

In [50]:
ex_vf = mediatools.VideoFile.from_path(ex_vid)
cmd = ex_vf.ffmpeg(
    input_args = mediatools.ffmpeg.FFInputArgs(
        ss=probe.start_time, 
        to=half_duration
    ),
    output = ffoutput(output_path, y=True)
)
cmd.get_command()

'ffmpeg -ss 0.000000 -to 16.7183675 -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -y /tmp/tmpn7bpy8zq/test_video_output.mp4'

The `run` the command will actually execute the ffmpeg call and return a `FFMPEGResult` instance with information about the command and `stderr`/`stdout`.

In [15]:
result = cmd.run()
result

FFMPEGResult(command=ffmpeg -ss 0.000000 -to 16.7183675 -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -y /tmp/tmpn7bpy8zq/test_video_output.mp4, returncode=0, output_length=4280)

For some unknown reason, ffmpeg pipes debugging information to stderr, which you can access through the `stderr` attribute.

In [16]:
print(result.stderr)

Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2avc1mp41
    encoder         : Lavf58.39.101
  Duration: 00:00:33.44, start: 0.000000, bitrate: 623 kb/s
  Stream #0:0[0x1](und): Video: h264 (Main) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 360x640 [SAR 1:1 DAR 9:16], 487 kb/s, 30 fps, 30 tbr, 15360 tbn (default)
    Metadata:
      handler_name    : ISO Media file produced by Google Inc.
      vendor_id       : [0][0][0][0]
  Stream #0:1[0x2](eng): Audio: aac (LC) (mp4a / 0x6134706D), 44100 Hz, stereo, fltp, 128 kb/s (default)
    Metadata:
      handler_name    : ISO Media file produced by Google Inc.
      vendor_id       : [0][0][0][0]
Stream mapping:
  Stream #0:0 -> #0:0 (h264 (native) -> h264 (libx264))
  Stream #0:1 -> #0:1 (aac (native) -> aac (native))
Press [q] to stop, [?] for help
[libx264 @ 0x5f13761ca940] using SAR=1/1
[libx264 @

You can verify that the command was successful by probing the new file. Notice the difference between the requested duration and the actual duration. This is an issue because videos are composed of discrete frames.

In [17]:
mediatools.ffmpeg.probe(output_path).duration

16.733333

## Creating Filterchains and Filtergraphs

Audio and video filters are an important feature of FFMPEG - they allow you to do things like crop, scale, and even create thumbnails from input videos.

There are two possible ways of constructing filters, both specified through the CLI as strings:

- **filterchain**: sequence of filters applied to an input video or audio stream. Specified by passing the arguments `-v:f` (for video filters) or `-a:f` (for audio filters).
- **filtergraph**: proper DAG describing transformations applied to an input video or audio stream. Specified by passing the argument `-filter_complex` (handles both audio and video).

In the `mediatools` ffmpeg interface, you can pass these as strings to the `v_f`, `a_f`, or `filter_complex` arguments of `ffoutput`. There are, however, several methods that make it easier to create properly structured filter specifications.

In [18]:
output_path = temp_path / "cropped_video.mp4"

ffmpeg(
    input=ffinput(str(ex_vid)),
    output=ffoutput(
        str(output_path), 
        v_f=f'scale=w=480:h=640', 
        y=True
    ),
).run()
mediatools.ffmpeg.probe(str(output_path)).video.resolution

(480, 640)

### Simple Filterchains

There are two methods that make it easy to construct simple filterchains to be passed as `v_f` (for video) or `a_f` (for audio):

- `filter_link`: specifies a single transformation of an input video or audio stream.
- `filterchain`: specifies a series of transformations for an input video or audio stream.

I will now show how these two can be used to construct filters.



#### `filter_link`

You can see how `filter_link` accepts the name of the filter and either positional or keyword arguments to specify the behavior.

In [19]:
from mediatools.ffmpeg import filter_link, filterchain

In [20]:
filter_link('scale', '480:640')

'scale=480:640'

In [21]:
filter_link('scale', w=480, h=640)

'scale=w=480:h=640'

In [22]:
filter_link('crop', w=100, h=100, x=100, y=100)

'crop=w=100:h=100:x=100:y=100'

#### `filterchain`

Use `filterchain` to chain filter links together. The output of this function is just a string, so it can be passed directly to the `v_f` argument.

In [23]:
filterchain(
    filter_link('crop', w=100, h=100, x=100, y=100),
    filter_link('scale', w=480, h=640),
)

'crop=w=100:h=100:x=100:y=100,scale=w=480:h=640'

As a second example, I create a filterchain that will extract animated thumbs from a video file. Here I wrap the filter in a parameterized factory function.

In [24]:
def animated_thumb_filter(pts: float, fps: int, width: int, height: int):
    return filterchain(
        filter_link('setpts', f'PTS/{pts}'),
        filter_link('fps', fps=fps),
        filter_link('scale', w=width, h=height, force_original_aspect_ratio='decrease'),
    )

### Complex Filtergraphs

There are two functions that can help you express complex filtergraph designs to be passed as `filter_complex`:

- `filtergraph_link`: specifies a single transformation from specified input and output streams.
- `filtergraph`: combines several filtergraph links to create a complete filtergraph.

In [25]:
from mediatools.ffmpeg import filtergraph_link, filtergraph

#### `filtergraph_link`

The `filtergraph_link` method is very similar to `filter_link` except that it adds input and output labels on either end of the string.

In [26]:
filtergraph_link('scale', w=480, h=640, input='0:v', output='scaled')

'[0:v]scale=w=480:h=640[scaled]'

The `filtergraph_link` function also allows you to provide multiple inputs or outputs using combinations of the `input`, `inputs`, `output`, and `outputs` parameters.

In [27]:
filtergraph_link('split', input='0:v', outputs=['bg_raw', 'fg_raw'])

'[0:v]split[bg_raw][fg_raw]'

In fact, you can even use it without input or output labels - it is a more general case of `filter_link`, which was created to have simpler arguments.

In [28]:
filtergraph_link('scale', w=480, h=640)

'scale=w=480:h=640'

Importantly, `filtergraph_link` can also accept filter links from `filter_link` and filterchains from `filterchain` instead of a filter name. This will save you from needing to construct additional filtergraph links.

In the example below, we create a filtergraph link from a filterchain of two links: one to scale and one to apply a gaussian blur.

In [29]:
chain = filterchain(
    filter_link("scale", w="ih", h="ih"),
    filter_link("gblur", sigma=25)
)

filtergraph_link(
    chain,
    input="bg_raw", 
    output="bg_ready"
)

'[bg_raw]scale=w=ih:h=ih,gblur=sigma=25[bg_ready]'

#### `filtergraph`

The `filtergraph` function allows you to construct a filtergraph that audio and video streams "flow" through to produce the final output. Each link is a transformation and the labels are intermediary points, so the order does not matter. Many filtergraphs essentially follow the pattern of processing each input stream in parallel and then overlaying them at the end, so you will see these examples.

The best way to see `filtergraph` in action is to look at examples.

#### `filtergraph` Example 1: Picture-in-picture

First, I create a filtergraph that simulates picture-in-picture using a single video: both the big and small videos will come from the same source. To do that, I split the input into two parts that will be eventually overlaid: the first remains the same, and the second one is scaled to be the size of the tiny pane. Finally, the tiny pane is overlaid back onto the output video. 

In [30]:
def picture_in_picture_self_filter(pip_scale_factor: int, margin: int):
    return filtergraph(
        # 1. Split the video
        filtergraph_link(
            'split', 
            input='0:v', 
            outputs=['main_raw', 'pip_raw']
        ), 
        
        # 2. Process the PiP pane (Scale it down)
        filtergraph_link(
            "scale", 
            # Using iw/4 and ih/4 keeps the exact same aspect ratio automatically
            w=f"iw/{pip_scale_factor}", 
            h=f"ih/{pip_scale_factor}", 
            input="pip_raw", 
            output="pip_ready"
        ), 
        
        # 3. Overlay the tiny pane onto the main video
        filtergraph_link(
            "overlay", 
            inputs=["main_raw", "pip_ready"], 
            # W = Main Width, w = PiP Width. 
            # (W - w) pushes it all the way to the right edge. Subtracting the margin brings it back slightly.
            x=f"W-w-{margin}", 
            # H = Main Height, h = PiP Height.
            # (H - h) pushes it all the way to the bottom edge. Subtracting the margin brings it up slightly.
            y=f"H-h-{margin}"
        )
    )

In [31]:
filtergraph(
    filtergraph_link('split', input='0:v', outputs=['base', 'pip_raw']), 
    filtergraph_link(
        filterchain(
            filter_link("scale", w="ih", h="ih"),
            filter_link("gblur", sigma=25),
            filter_link('crop', w=100, h=100, x=100, y=100),
        ), 
        input="pip_raw", 
        output="pip_final"
    ), 
    filtergraph_link(
        "overlay", 
        inputs=["bg_ready", "fg_raw"], 
        x="(W-w)/2", 
        y="(H-h)/2"
    )
)

'[0:v]split[base][pip_raw];[pip_raw]scale=w=ih:h=ih,gblur=sigma=25,crop=w=100:h=100:x=100:y=100[pip_final];[bg_ready][fg_raw]overlay=x=(W-w)/2:y=(H-h)/2'

#### `filtergraph` Example 2: Blurred Video Padding

This example shows how to create a filtergraph that transforms the input into a new video with symmetric aspect ratio where padding is filled in with a blurred version of the original video.

In [32]:
def blurred_padding_symetric_filter(blur_sigma: int = 25):
    return filtergraph(
        
        # 1. Split the input video stream into two parts: `bg_raw` and `fg_raw`.
        filtergraph_link('split', input='0:v', outputs=['bg_raw', 'fg_raw']), 
        
        # 2. Scale `bg_raw` to the input height (`ih`) and width (`iw`), then apply a gaussian blur to produce `bg_ready`.
        filtergraph_link(
            filterchain(
                filter_link("scale", w="ih", h="ih"),
                filter_link("gblur", sigma=blur_sigma)
            ), 
            input="bg_raw", 
            output="bg_ready"
        ), 

        # 3. Overlay `fg_raw` on top of `bg_ready` according to the centering math: `x="(W-w)/2", y="(H-h)/2"` where:
        #    - `x` and `y` are the starting coordinates of the overlay.
        #    - `W` and `H` are the width and height of the main/background video
        #    - `w` and `h`  are the width and height of the foreground video.
        filtergraph_link(
            "overlay", 
            inputs=["bg_ready", "fg_raw"], 
            x="(W-w)/2", 
            y="(H-h)/2"
        )
    )

Now say we want to specify a custom resolution for the output so we can dynamically set the blurred output padding accoridng to the original and desired resolutions.

In [33]:
def blurred_padding_filter(target_w: int, target_h: int, blur_sigma: int = 25):
    return filtergraph(

        # 1. Split the input video stream into two parts: `bg_raw` and `fg_raw`.
        filtergraph_link(filter_spec='split', input='0:v', outputs=['bg_raw', 'fg_raw']), 

        # 2. Fill the frame completely without stretching or leaving black bars
        filtergraph_link(
            filter_spec = filterchain(
                
                # 2a. Scale `bg_raw` to fill the frame while preserving aspect ratio, which may cause some cropping.
                filter_link(
                    "scale", 
                    w=target_w, 
                    h=target_h, 
                    force_original_aspect_ratio="increase"
                ),
                
                # 2b. Crop the scaled `bg_raw` to the target dimensions.
                filter_link("crop", w=target_w, h=target_h),
                
                # 2c. Apply a gaussian blur to the cropped background.
                filter_link("gblur", sigma=blur_sigma)
            ), 
            input="bg_raw", 
            output="bg_ready"
        ), 
        
        # 3. Scale `fg_raw` to fit within the target dimensions while preserving aspect ratio, which may cause letterboxing/pillarboxing.
        filtergraph_link(
            filter_spec = "scale", 
            w=target_w, 
            h=target_h, 
            force_original_aspect_ratio="decrease",
            input="fg_raw",
            output="fg_ready"
        ),

        # 4. Overlay `fg_ready` on top of `bg_ready` according to the centering math: `x="(W-w)/2", y="(H-h)/2"`.
        #    - `x` and `y` are the starting coordinates of the overlay.
        #    - `W` and `H` are the width and height of the main/background video
        #    - `w` and `h`  are the width and height of the foreground video.
        filtergraph_link(
            filter_spec="overlay", 
            inputs=["bg_ready", "fg_ready"], 
            x="(W-w)/2", 
            y="(H-h)/2"
        )
    )
blurred_padding_filter(target_w=480, target_h=640, blur_sigma=25)

'[0:v]split[bg_raw][fg_raw];[bg_raw]scale=w=480:h=640:force_original_aspect_ratio=increase,crop=w=480:h=640,gblur=sigma=25[bg_ready];[fg_raw]scale=w=480:h=640:force_original_aspect_ratio=decrease[fg_ready];[bg_ready][fg_ready]overlay=x=(W-w)/2:y=(H-h)/2'

## Common use-cases

While ffmpeg is an extremely robust and flexible library, a small set of basic recipes may be used to solve the majority of challenges.

### 1. Splicing Videos

Splice videos by providing the `ss` and `to` parameters to the input.

In [34]:
result = ffmpeg(
    input=ffinput(ex_vid, ss=0, to=1.0),
    output=ffoutput(temp_path / "spliced.mp4"),
).run()
result.command.get_command()

'ffmpeg -ss 0 -to 1.0 -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -n /tmp/tmpn7bpy8zq/spliced.mp4'

### 2. Cropping Videos
Crop a video using the ffmpeg `crop` filter.

In [35]:
ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(
        temp_path / "filtered_output.mp4",
        v_f=filter_link(
            'crop',
            w=200, # width
            h=100, # height
            x=100, # topleft point x-axis
            y=50, # topleft point y-axis
        ),
    ),
).run()
result.command.get_command()

'ffmpeg -ss 0 -to 1.0 -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -n /tmp/tmpn7bpy8zq/spliced.mp4'

### 3. Compressing Videos

You can compress videos either by specifying the Constant Rate Factor (CRF) or bitrate (with a two-pass system).

#### Compress by Specifying CRF

Specifying the Constant Rate Factor (CRF) of a video is the easiest way to describe desired compression. This is a quick guide to various CRF values:

| Quality Level | CRF Range | Visual Description & Artifacts | Relative File Size & Data Impact |
| :--- | :--- | :--- | :--- |
| **Lossless** | $0$ | Mathematically identical to the source. No compression artifacts. | **Extremely Large.** Can be $10\times$ to $100\times$ larger than standard encodes. |
| **Near-Lossless** | $1-16$ | Imperceptible loss. Used when the source is exceptionally high quality or for intermediate renders. | **Very Large.** High bitrates that may exceed the playback capabilities of some hardware. |
| **High Quality** | $17-19$ | **Visually Transparent.** The human eye cannot distinguish this from the source in a blind test. | **Large.** Significant reduction from lossless, but still carries a heavy data load. |
| **Standard** | $20-23$ | Excellent clarity. Minor "mosquito noise" or slight softening may appear in extremely high-motion or dark scenes. | **Moderate.** The "sweet spot" for balancing high-definition visuals with manageable storage. |
| **Efficient** | $24-28$ | Discernible compression. Fine textures (like skin pores or grain) begin to blur; some "blocking" in shadows. | **Small.** Roughly $50\%$ the size of a CRF $18$ encode. Ideal for streaming and mobile devices. |
| **Low Quality** | $29-35$ | Obvious artifacts. Heavy color banding in gradients and "shimmering" around moving edges. | **Very Small.** Drastic reduction in bitrate; usable only when storage is at a premium. |
| **Extreme** | $36-51$ | Severe degradation. The image becomes "muddy" with heavy pixelation and lost detail. | **Tiny.** Lowest possible data footprint; quality is generally considered unacceptable for viewing. |

In [36]:
p = temp_path / "compressed.mp4"
ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(p, crf=17, c_v="libx264", y=True),
).run()
result.command.get_command()

'ffmpeg -ss 0 -to 1.0 -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -n /tmp/tmpn7bpy8zq/spliced.mp4'

#### Compress by Specifying Bitrate

You may also specify bitrate with a two-pass approach. This is a description of the two passes:

1. In this stage, the encoder (e.g., `libx264`) performs a full-length analysis of the source video. It doesn't focus on high-fidelity output; instead, it records metadata about the **spatial and temporal complexity** of every frame into a temporary log file (`ffmpeg2pass-0.log`).

    * **Complexity Mapping:** It identifies which frames are "expensive" (high detail/fast motion) and which are "cheap" (static backgrounds/flat colors).
    * **Frame Type Selection:** It pre-calculates the optimal placement of **I-frames** (keyframes), **P-frames**, and **B-frames**.
    * **The Output:** Because the goal is data gathering, the video output is discarded (sent to `/dev/null` or `NUL`) to save I/O time.


2. The encoder restarts the process, but this time it reads the log file from Pass 1. It treats the `target bitrate` (e.g., `2000k`) as a **global average budget** rather than a strict limit for every second.

    * **Bit Redistribution:** It "borrows" bits from the "cheap" frames (lowering their bitrate below 2000k) and "lends" them to the "expensive" frames (allowing them to spike well above 2000k).
    * **Consistent Quality:** This results in a video with **uniform visual quality** throughout, whereas a 1-pass encode at the same bitrate would look great in slow scenes but fall apart (pixelate) during action scenes.


In [37]:
current_bitrate = mediatools.ffmpeg.probe(ex_vid).bit_rate
target_bitrate = current_bitrate / 2
f'{current_bitrate/1000000} Mbps, {target_bitrate/1000000} Mbps'

'0.62379 Mbps, 0.311895 Mbps'

In [38]:
tmp_log_dir = temp_path / "ffmpeg_logs"
result = ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(
        '/dev/null',
        c_v='libx264',
        b_v=target_bitrate,
        an=True,
        f='null',
        pass_=1,
        passlogfile=str(tmp_log_dir),
        y=True,
    ),
).run()
result.command.get_command()

'ffmpeg -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -c:v libx264 -b:v 311895.0 -f null -pass 1 -passlogfile /tmp/tmpn7bpy8zq/ffmpeg_logs -an -y /dev/null'

And you can see it generated this log file:

In [39]:
list(temp_path.rglob(f'**/*.log'))

[PosixPath('/tmp/tmpn7bpy8zq/ffmpeg_logs-0.log')]

In [40]:
result = ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(
        temp_path / "compressed_bitrate.mp4",
        c_v='libx264',
        b_v=target_bitrate,
        c_a='copy',
        y=True,
        passlogfile=str(tmp_log_dir),
        pass_=2,
    ),
).run()
result.command.get_command()

'ffmpeg -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -c:v libx264 -b:v 311895.0 -c:a copy -pass 2 -passlogfile /tmp/tmpn7bpy8zq/ffmpeg_logs -y /tmp/tmpn7bpy8zq/compressed_bitrate.mp4'

And you can see that the output is much closer to the target bitrate.

In [41]:
mediatools.ffmpeg.probe(result.output_file).bit_rate

451853

### 4. Creating Thumbnails

There are two primary types of thumnails you may want to use: static thumbnails and animated thumbnails.

#### Static Thumbnails

Create static thumbnails using a `scale` filter in conjunction with specifying with `vframes=1` and `ss` to the time point when the thumb should be created.

In [42]:
result = ffmpeg(
    input=ffinput(ex_vid, ss=1.0),
    output=ffoutput(
        temp_path / "thumbnail.jpg",    
        v_f=filter_link('scale', width=200, height=300), 
        vframes=1,
        #y=True,
    ),
).run()
result.command.get_command()

'ffmpeg -ss 1.0 -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -vframes 1 -vf scale=width=200:height=300 -n /tmp/tmpn7bpy8zq/thumbnail.jpg'

In [43]:
result.output_file.exists()

True

#### Animated Thumbnails with Presentation Time Stamp (PTS)

You can create an animated thumbnail by integrating `setpts` (number of points to sample) and `fps` filters. 


**Presentation Time Stamp (PTS) is a number assigned to every frame that tells the player exactly when to show that frame relative to the start of the video.

For example, in the filter string `setpts=PTS/2,fps=fps=2`:

* **`PTS`**: Represents the original "real-world" time of the frame. If a frame is at the 10-second mark, its PTS reflects 10 seconds.
* **`setpts=PTS/2`**: You are mathematically manipulating time. Dividing the timestamp by 2 tells FFmpeg: "Take the frame that was supposed to appear at 10 seconds and show it at 5 seconds instead."
    * **Result:** The video plays at **2x speed**.
* **`fps=fps=2`**: This is your "sampling rate." After you've sped up the motion, you are telling FFmpeg to only keep 2 frames for every 1 second of output time.

In other words, `setpts=PTS` -> 1x (no change), `setpts=PTS/2` -> 2x speed, and `setpts=PTS/0.5` -> 0.5x speed.

**How to Calculate for a Specific Thumbnail Length**
If you want your animated GIF to represent a specific "slice" of time, you have to coordinate the `setpts` with your `fps`. If you want to turn **X** seconds of real video into **Y** seconds of GIF:

$$\text{setpts} = \text{PTS} \times \left( \frac{Y}{X} \right)$$

For example, say you want to condense **30 seconds** of footage into a **3-second** animated thumbnail:
1.  $3 / 30 = 0.1$
2.  Your filter: `setpts=0.1*PTS,fps=10` (This gives you a 3-second GIF at 10fps).

**Note**: when you use `setpts=PTS/2`, you are crunching the timeline. If you don't use the `fps` filter afterward, FFmpeg might try to keep all the original frames, leading to a very high-bitrate GIF that chokes browsers. By putting `fps=2` **after** the speed change, you are effectively saying: "Now that I've sped this up, give me a snapshot every half-second of the *new* timeline."


In [44]:
pts = 2
fps = 2
width = 100
height = 100

result = ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(
        temp_path / "animated_thumbnail.gif", 
        v_f=filterchain(
            filter_link('setpts', f'PTS/{pts}'),
            filter_link('fps', fps=fps),
            filter_link('scale', w=width, h=height, force_original_aspect_ratio='decrease'),
        ),
        y=True,
    ),
).run()
result.command.get_command()

'ffmpeg -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -vf setpts=PTS/2,fps=fps=2,scale=w=100:h=100:force_original_aspect_ratio=decrease -y /tmp/tmpn7bpy8zq/animated_thumbnail.gif'

## Built-in Filters and Functions

This package also has several pre-built filtergraphs and command functions to accomplish common tasks. 

### Built-in Filtergraph: Blurred Padding

The filter graph for blurred padding can be used by passing it directly to `filter_complex` or by including it in a larger, more complex filtergraph.

In [45]:
from mediatools.ffmpeg import filtergraph_blurred_padding

width, height = 500, 500
blur_sigma = 10

result = ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(
        temp_path / "resize_with_blur.mp4", 
        filter_complex=filtergraph_blurred_padding(
            input="0:v",
            target_w=width,
            target_h=height,
            blur_sigma=blur_sigma,
        ),
        y=True,
    ),
).run()
result.command.get_command()

"ffmpeg -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -filter_complex '[0:v]split[bg_raw][fg_raw];[bg_raw]scale=w=500:h=500:force_original_aspect_ratio=increase,crop=w=500:h=500,gblur=sigma=10[bg_ready];[fg_raw]scale=w=500:h=500:force_original_aspect_ratio=decrease[fg_ready];[bg_ready][fg_ready]overlay=x=(W-w)/2:y=(H-h)/2' -y /tmp/tmpn7bpy8zq/resize_with_blur.mp4"

### Built-in Filtergraph: Animated Thumbs

The `filtergraph_animated_thumb` function returns a filtergraph that optionally builds on the blurred padding filter if desired.

In [46]:
from mediatools.ffmpeg import filtergraph_animated_thumb
width, height = 400, 400
fps = 2
pts = 10

result = ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(
        temp_path / "animated_thumbnail.gif", 
        filter_complex=filtergraph_animated_thumb(
            input="0:v",
            target_w=width,
            target_h=height,
            fps=fps,
            pts=pts,
            force_original_aspect_ratio='decrease',
        ),
        y=True,
    ),
).run()
result.command.get_command()

"ffmpeg -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -filter_complex '[0:v]setpts=PTS/10,fps=fps=2,scale=w=400:h=400:force_original_aspect_ratio=decrease' -y /tmp/tmpn7bpy8zq/animated_thumbnail.gif"

In [47]:
from mediatools.ffmpeg import filtergraph_animated_thumb
width, height = 400, 400
blur_sigma = 10
fps = 2
pts = 10

result = ffmpeg(
    input=ffinput(ex_vid),
    output=ffoutput(
        temp_path / "animated_thumbnail_with_blur.gif", 
        filter_complex=filtergraph_animated_thumb(
            input="0:v",
            target_w=width,
            target_h=height,
            use_blurred_padding=True,
            blur_sigma=blur_sigma,
            fps=fps,
            pts=pts,
        ),
        y=True,
    ),
).run()
result.command.get_command()

"ffmpeg -i /tmp/tmpn7bpy8zq/totk_builds/op_builds.mp4 -hide_banner -nostats -filter_complex '[0:v]setpts=PTS/10,fps=fps=2[sampled];[sampled]split[bg_raw][fg_raw];[bg_raw]scale=w=400:h=400:force_original_aspect_ratio=increase,crop=w=400:h=400,gblur=sigma=10[bg_ready];[fg_raw]scale=w=400:h=400:force_original_aspect_ratio=decrease[fg_ready];[bg_ready][fg_ready]overlay=x=(W-w)/2:y=(H-h)/2' -y /tmp/tmpn7bpy8zq/animated_thumbnail_with_blur.gif"